# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Abstract

Which pages should a content team refresh first? We built a scoring model on 30,000 pseudonymized pages across 32 FlyRank clients, using a 90-day trailing window of search performance data. A transparent rule-based baseline and a Random Forest regressor were evaluated on a client-grouped train/test split to prevent leakage, with Random Forest achieving roughly 4x the naive base rate at precision@20 — a meaningful but imperfect improvement, since the model has no awareness of commercial value (`cpc`/`search_volume` were excluded to prevent label leakage). The result is two decision-support queues — one value-aware but narrow in scope, one broader but value-blind — designed to help editors prioritize limited refresh capacity, not to replace their judgment or predict outcomes.

## 1. Question

*The research question and the decision it supports.*

### Which pages should a content team refresh first? 
We built a scoring model that ranks webpages by combining how much the web pages are declining in search performance with how commercially valuable they are, so FlyRank's content/SEO editors can prioritize limited refresh time on the pages most worth fixing. A wrong recommendation costs wasted editor hours on a low-value page, or worse, erodes the team's trust in the system.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

1. Source: The FlyRank internship dataset - 30,000 rows, one row per pseudonymized page, 32 clients, trailing 90-day window.
2. Exclusions: impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d (window overlaps the label's trend calculation); cpc, search volume (Direct multiplicative components of target(refresh_score)),  sessions_prev_30d, trend_pct, trend_direction (label driven); provider_used, model_used (unreliable metadata, per data contract)

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Assumptions
This work rests on several unproven assumptions, stated openly:
1. refresh_score - A weighted combination of decline and commercial value is a reasonable proxy for editorial priority, though it was never validated against actual editor judgment.
2. A single 90-day snapshot is representative enough to rank pages, with no seasonality or multi-period check performed.
3. The top 10% is_priority cutoff reflects realistic editor capacity(roughly 20-50 pages reviewed per cycle), based on a stated assumption rather than a measured team bandwidth.
4. Client-level differences are the dominant hidden confound worth guarding against, which is why every split in this project is grouped by client_id rather than random.

Features
'competition', 'competition_level', 'content_type', 'main_intent',
'word_count', 'char_count', 'impressions_90d', 'clicks_90d',
'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d',
'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
'days_with_sessions', 'content_age_days', 'age_tier_order',
'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
'scroll_rate', 'ai_traffic_pct', 'word_count_missing',
'char_count_missing'

Label definition
We define refresh_score = trend_pct_dampened * cpc * search_volume, computed only for pages with a valid trend signal. is_priority is a binary label: 1, if a page's refresh_score falls is the top 10% of that filtered population, 0 otherwise. This is a defined proxy, not an observed outcome. No page in this dataset has a confirmed real-world 'needed refresh' label.



## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 4. Results (vs baseline)

**Comparison table** (grouped split, test set, precision@k):

| Method | precision@20 | Note |
|---|---|---|
| Base rate (random) | 0.113 | Naive floor |
| Baseline rule | 1.0 | Invalid — circular with `is_priority`, see Methodology |
| Linear Regression | 0.20 | ~1.8x base rate |
| Random Forest | 0.45 | ~4x base rate |

Random Forest is the strongest validated method, at roughly 4x the naive floor. Linear Regression shows a modest lift. The baseline's perfect score is explicitly excluded from this comparison for the reasons stated in Methodology — base rate, not the baseline rule, is the fair benchmark here.

![precision@20 comparison](charts/precision_comparison.png)

**Error analysis**

Of Random Forest's top 20 predictions, 11 were misranked (`is_priority = 0`). Manually reviewing these cases revealed a consistent pattern: pages with very low `cpc` (often <$0.15) and recent update dates were still ranked at the top.

Root cause: the model has no commercial-value signal. `cpc` and `search_volume` were correctly excluded as label-derived leakage, but this leaves the model structurally unable to distinguish a high-value declining page from a low-value one. Feature importance confirms this — the top 10 features are entirely decline/engagement signals (`avg_position`, `content_age_days`, `sessions_90d`, etc.); no value-adjacent feature appears anywhere in the top 10.

![feature importance](charts/feature_importance.png)

## 5. Limitations

*What this work cannot claim.*

## 5. Limitations

This work cannot claim more than what the evidence supports.

**1. No commercial-value awareness in the model.**
`cpc` and `search_volume` were correctly excluded from the feature set as label-derived leakage, but this leaves Random Forest structurally unable to distinguish a high-value declining page from a low-value one. Confirmed by both error analysis (low-cpc pages misranked as top priority) and feature importance (no value-adjacent feature in the top 10).

**2. `is_priority` is a defined proxy, not an observed outcome.**
It was computed as the top 10% of `refresh_score` — no page in this dataset has a confirmed real-world "needed a refresh" label. A page appearing in either queue means it scored high on a formula, not that a human or business context has confirmed it as genuinely valuable to fix. The formula also cannot see content quality — a well-written page and a poorly-written page with identical metrics score identically.

**3. No causal claim is supportable.**
This dataset is a single 90-day cross-sectional snapshot; no page's before/after outcome from an actual refresh is recorded anywhere. This work can flag pages worth reviewing based on observed patterns (declining, stale, near-miss), but cannot claim that refreshing a listed page will cause traffic or ranking to improve. Any real-world improvement depends on execution quality, competitive dynamics, and search algorithm behavior — none of which this data measures.

**4. The baseline comparison in Results is not independent.**
The Week-4 baseline rule shares core formula ingredients with `is_priority`, making a precision@k comparison between them circular; base rate is the fair benchmark used instead (see Methodology).

**5. Random Forest's ~4x lift over base rate is a measured improvement, not evidence of reliable prediction.**
The model still misranked more than half of its own top-20 picks (11 of 20).

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 6. Ranked recommendations

Rather than a single merged ranking, this work produces two separate queues — because the baseline rule and Random Forest model optimize for genuinely different things, and forcing them into one list would hide that from the editor.

**Rule-Based Queue** — value-aware, narrow scope.
Restricted to the near-miss zone (`avg_position` 8-20), weighted by `cpc × search_volume`. Best for editors prioritizing commercial impact.

**Model-Based Queue** — broader scope, value-blind.
Scores across the full page population using decline and engagement patterns, but cannot see commercial value. Best for surfacing pages outside the narrow near-miss window that a rule-based approach would never consider.

Testing confirmed these genuinely diverge: even restricted to the same eligible pool (near-miss zone only), the two queues overlap on just 2 of their top 50 picks, and the Rule-Based Queue's picks average 2.5x higher `cpc` (3.33 vs 1.30) than the Model-Based Queue's. This is not noise — it's the direct, structural consequence of one method seeing value and the other not.

**Before acting on either queue, an editor should:**

1. Check `cpc`/`search_volume` for Model-Based Queue picks — the model didn't.
2. Open and read the actual page — neither method can detect existing content quality.
3. Never delete or unpublish a page based on this output alone — the queues were validated for flagging review candidates, not for higher-stakes removal decisions.

![reason code distribution](charts/reason_code_distribution.png)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

All charts referenced in this paper are generated below and saved to `../outputs/charts/`. Re-running this notebook top to bottom reproduces every figure exactly.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor

# --- 1. Load + build target (Weeks 1/5) ---
df = pd.read_csv(r'C:\Users\hamto\OneDrive\Desktop\Flyrank-Machine-Learning-Internship\data\raw\content_refresh_anonymized.csv')
df = df[df['search_volume'].notna() & df['cpc'].notna()]
df['trend_pct_flipped'] = -df['trend_pct']
lower = df['trend_pct_flipped'].quantile(0.01)
upper = df['trend_pct_flipped'].quantile(0.99)
df['trend_pct_dampened'] = df['trend_pct_flipped'].clip(lower, upper)
df['refresh_score'] = df['trend_pct_dampened'] * df['search_volume'] * df['cpc']
df = df[df['trend_pct'].notna()]
threshold = df['refresh_score'].quantile(0.90)
df['is_priority'] = (df['refresh_score'] >= threshold).astype(int)

# --- 2. Baseline rule (Week 4, on the FULL population, unchanged) ---
value = df['cpc'] * df['search_volume']
declining_rank = df['trend_pct_dampened'].abs().rank(pct=True)
staleness_rank = df['days_since_last_update'].rank(pct=True)

near_miss_mask = df['avg_position'].between(8, 20)
decline_threshold = df.loc[near_miss_mask, 'trend_pct_dampened'].quantile(0.75)
stale_threshold = 200

declining_only_mask = near_miss_mask & (df['trend_pct_dampened'] >= decline_threshold) & (df['days_since_last_update'] < stale_threshold)
stale_only_mask = near_miss_mask & (df['days_since_last_update'] >= stale_threshold) & (df['trend_pct_dampened'] < decline_threshold)
stale_declining_mask = near_miss_mask & (df['trend_pct_dampened'] >= decline_threshold) & (df['days_since_last_update'] >= stale_threshold)

score_declining = value * declining_rank
score_stale = value * staleness_rank
score_stale_declining = value * (declining_rank + staleness_rank)

conditions = [stale_declining_mask, declining_only_mask, stale_only_mask]
score_choices = [score_stale_declining, score_declining, score_stale]
reason_choices = ['NEAR_MISS_STALE_DECLINING', 'NEAR_MISS_DECLINING', 'NEAR_MISS_STALE']

df['baseline_score'] = np.select(conditions, score_choices, default=0)
df['reason_code'] = np.select(conditions, reason_choices, default=None)

# --- 3. Grouped split (Week 5) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

# --- 4. Build features + fit RF (Week 5) ---
drop_cols = [
    'client_id', 'content_id', 'is_priority', 'trend_pct', 'trend_pct_flipped',
    'trend_pct_dampened', 'refresh_score', 'cpc', 'search_volume',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'trend_direction',
    'provider_used', 'model_used', 'age_tier', 'impression_tier', 'position_tier',
    'freshness_tier', 'char_count_tier', 'word_count_tier',
    'baseline_score', 'reason_code',
]
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['refresh_score']
X_test = test_df.drop(columns=drop_cols)

for col in ['word_count', 'char_count']:
    X_train[f'{col}_missing'] = X_train[col].isna().astype(int)
    X_test[f'{col}_missing'] = X_test[col].isna().astype(int)
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

for col in ['competition', 'scroll_rate']:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

for col in ['competition_level', 'main_intent']:
    X_train[col] = X_train[col].fillna('missing')
    X_test[col] = X_test[col].fillna('missing')

X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_encoded, y_train)
test_df = test_df.copy()
test_df['rf_pred'] = rf.predict(X_test_encoded)

In [4]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import os

os.makedirs('../outputs/charts', exist_ok=True)

# 1. Precision comparison
methods = ['Base rate', 'Linear Regression', 'Random Forest']
values = [0.113, 0.20, 0.45]
plt.figure(figsize=(6,4))
plt.bar(methods, values, color=['gray', 'steelblue', 'darkorange'])
plt.ylabel('precision@20')
plt.title('precision@20 vs base rate (grouped split)')
plt.savefig('../outputs/charts/precision_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

# 2. Feature importance (reuse your fitted `rf` and `X_train_encoded` from Section 3)
importances = pd.Series(rf.feature_importances_, index=X_train_encoded.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(7,5))
plt.barh(importances.index[::-1], importances.values[::-1], color='darkorange')
plt.xlabel('Feature importance')
plt.title('Top 10 Random Forest Feature Importances')
plt.tight_layout()
plt.savefig('../outputs/charts/feature_importance.png', dpi=150)
plt.close()

# 3. Reason code distribution (reuse `df['reason_code']` from your baseline)
counts = df['reason_code'].value_counts()
plt.figure(figsize=(6,4))
plt.bar(counts.index, counts.values, color='steelblue')
plt.ylabel('Number of pages')
plt.title('Baseline Reason Code Distribution')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('../outputs/charts/reason_code_distribution.png', dpi=150)
plt.close()

# 4. Age pattern (CTR proxy — no composite health score in this dataset)
df['age_bucket'] = pd.cut(df['content_age_days'], bins=[0,30,90,180,270,365,10000],
labels=['0-30','31-90','91-180','181-270','271-365','365+'])
age_pattern = df.groupby('age_bucket', observed=True)['ctr'].mean()
plt.figure(figsize=(6,4))
plt.plot(age_pattern.index.astype(str), age_pattern.values, marker='o', color='darkorange')
plt.ylabel('Mean CTR')
plt.xlabel('Content age bucket (days)')
plt.title('CTR by Content Age (proxy — no composite health score in this dataset)')
plt.tight_layout()
plt.savefig('../outputs/charts/age_pattern.png', dpi=150)
plt.close()

print("All 4 charts saved to ../outputs/charts/")

All 4 charts saved to ../outputs/charts/


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.